# Prediciton data by trimming most recent weather data to match with elexon data

In [1]:
import requests
import pandas as pd


OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
ELEXON_URL = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"


def get_aligned_weather_elexon():
    # -----------------------------
    # 1) Weather
    # -----------------------------
    weather_params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation",
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "past_days": 7,
        "wind_speed_unit": "ms",
        "current": "temperature_2m",
    }

    weather_res = requests.get(OPEN_METEO_URL, params=weather_params, timeout=30)
    weather_res.raise_for_status()
    weather_data = weather_res.json()

    weather_df = pd.DataFrame(weather_data["hourly"])
    weather_df["time"] = pd.to_datetime(weather_df["time"])

    # wind 100m
    weather_df["wind_speed_100m"] = (
        weather_df["wind_speed_120m"] + weather_df["wind_speed_80m"]
    ) / 2
    weather_df = weather_df.drop(columns=["wind_speed_120m", "wind_speed_80m"])

    forecast_start = pd.to_datetime(weather_data["current"]["time"])

    # hourly → half-hourly
    weather_df = weather_df.set_index("time").resample("30min").interpolate("time").reset_index()

    weather_hist = weather_df[weather_df["time"] < forecast_start].copy()

    # -----------------------------
    # 2) Elexon
    # -----------------------------
    elexon_params = {
        "from": (forecast_start - pd.Timedelta(days=8)).strftime("%Y-%m-%dT%H:%MZ"),
        "to": forecast_start.strftime("%Y-%m-%dT%H:%MZ"),
        "format": "json",
    }

    elexon_res = requests.get(ELEXON_URL, params=elexon_params, timeout=30)
    elexon_res.raise_for_status()
    elexon_json = elexon_res.json()

    records = []
    for period in elexon_json["data"]:
        for item in period["data"]:
            records.append({
                "startTime": period["startTime"],
                "psrType": item["psrType"],
                "quantity": item["quantity"],
            })

    elexon_df = pd.DataFrame(records)

    elexon_df["startTime"] = pd.to_datetime(elexon_df["startTime"], utc=True).dt.tz_localize(None)

    elexon_df = (
        elexon_df.pivot_table(
            index="startTime",
            columns="psrType",
            values="quantity",
            aggfunc="sum",
        )
        .sort_index()
        .reset_index()
    )

    elexon_df.columns.name = None

    # total output
    fuel_cols = [c for c in elexon_df.columns if c != "startTime"]
    elexon_df["total_output_MW"] = elexon_df[fuel_cols].sum(axis=1)

    # -----------------------------
    # 3) Align to same 336 timestamps
    # -----------------------------
    common_end = min(
        weather_hist["time"].max(),
        elexon_df["startTime"].max()
    )

    common_end = pd.Timestamp(common_end).floor("30min")
    common_start = common_end - pd.Timedelta(minutes=30 * (336 - 1))

    common_index = pd.date_range(common_start, common_end, freq="30min")

    weather_hist_df = (
        weather_hist.set_index("time")
        .reindex(common_index)
        .rename_axis("time")
        .reset_index()
    )

    elexon_hist_df = (
        elexon_df.set_index("startTime")
        .reindex(common_index)
        .rename_axis("startTime")
        .reset_index()
    )

    return weather_hist_df, elexon_hist_df

In [2]:
weather_df, elexon_df = get_aligned_weather_elexon()

print(weather_df.shape)  # (336, ...)
print(elexon_df.shape)  # (336, ...)

(336, 10)
(336, 13)


In [3]:
weather_df

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-10 21:30:00,10.30,10.90,100.0,0.0,0.0,0.0,1010.40,0.0,9.3325
1,2026-03-10 22:00:00,10.50,11.00,100.0,0.0,0.0,0.0,1010.10,0.0,9.5750
2,2026-03-10 22:30:00,10.50,11.15,100.0,0.0,0.0,0.0,1009.50,0.1,10.0800
3,2026-03-10 23:00:00,10.50,11.30,100.0,0.0,0.0,0.0,1008.90,0.2,10.5850
4,2026-03-10 23:30:00,10.35,11.70,100.0,0.0,0.0,0.0,1008.45,0.5,10.9075
...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 19:00:00,13.80,3.80,0.0,0.0,0.0,0.0,1014.70,0.0,5.2600
332,2026-03-17 19:30:00,13.60,4.10,0.0,0.0,0.0,0.0,1014.75,0.0,5.0450
333,2026-03-17 20:00:00,13.40,4.40,0.0,0.0,0.0,0.0,1014.80,0.0,4.8300
334,2026-03-17 20:30:00,13.25,4.60,0.0,0.0,0.0,0.0,1014.95,0.0,4.9125


In [4]:
elexon_df

,startTime,Biomass,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
0,2026-03-10 21:30:00,2161.0,5446.0,0.0,0.0,2.0,322.0,2850.0,410.0,0.0,11698.565,10387.659,33277.224
1,2026-03-10 22:00:00,2133.0,5397.0,0.0,0.0,2.0,306.0,2849.0,546.0,0.0,11718.671,10210.152,33161.823
2,2026-03-10 22:30:00,1732.0,4730.0,0.0,0.0,2.0,293.0,2848.0,423.0,0.0,11758.889,10090.805,31877.694
3,2026-03-10 23:00:00,1731.0,4533.0,0.0,0.0,0.0,251.0,2851.0,189.0,0.0,11868.229,10187.685,31610.914
4,2026-03-10 23:30:00,1730.0,3932.0,0.0,0.0,0.0,245.0,2854.0,492.0,0.0,11978.611,10065.052,31296.663
...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 19:00:00,2706.0,8918.0,0.0,0.0,1132.0,326.0,3633.0,1136.0,0.0,8204.405,8289.768,34345.173
332,2026-03-17 19:30:00,2710.0,8672.0,0.0,0.0,857.0,341.0,3628.0,955.0,0.0,8059.434,8236.778,33459.212
333,2026-03-17 20:00:00,2736.0,8984.0,0.0,0.0,292.0,336.0,3628.0,812.0,0.0,8000.875,7989.156,32778.031
334,2026-03-17 20:30:00,2760.0,8885.0,0.0,0.0,0.0,330.0,3632.0,472.0,0.0,8249.009,7850.069,32178.078


# matching elexon data to most recent weather data

In [5]:
import requests
import pandas as pd


OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
ELEXON_URL = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"


def get_aligned_weather_elexon_fill():
    # -----------------------------
    # 1) Weather
    # -----------------------------
    weather_params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation",
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "past_days": 7,
        "wind_speed_unit": "ms",
        "current": "temperature_2m",
    }

    weather_res = requests.get(OPEN_METEO_URL, params=weather_params, timeout=30)
    weather_res.raise_for_status()
    weather_data = weather_res.json()

    weather_df = pd.DataFrame(weather_data["hourly"])
    weather_df["time"] = pd.to_datetime(weather_df["time"])

    weather_df["wind_speed_100m"] = (
        weather_df["wind_speed_120m"] + weather_df["wind_speed_80m"]
    ) / 2
    weather_df = weather_df.drop(columns=["wind_speed_120m", "wind_speed_80m"])

    forecast_start = pd.to_datetime(weather_data["current"]["time"])

    # hourly -> half-hourly
    weather_df = (
        weather_df.set_index("time")
        .resample("30min")
        .interpolate(method="time")
        .reset_index()
    )

    # keep exactly the same historical weather window as before
    forecast_idx = weather_df[weather_df["time"] >= forecast_start].index[0]
    start_idx = max(0, forecast_idx - 336)
    weather_hist_df = weather_df.iloc[start_idx:forecast_idx].reset_index(drop=True)

    weather_index = weather_hist_df["time"]

    # -----------------------------
    # 2) Elexon
    # -----------------------------
    elexon_params = {
        "from": weather_index.min().strftime("%Y-%m-%dT%H:%MZ"),
        "to": weather_index.max().strftime("%Y-%m-%dT%H:%MZ"),
        "format": "json",
    }

    elexon_res = requests.get(ELEXON_URL, params=elexon_params, timeout=30)
    elexon_res.raise_for_status()
    elexon_json = elexon_res.json()

    records = []
    for period in elexon_json["data"]:
        for item in period["data"]:
            records.append({
                "startTime": period["startTime"],
                "psrType": item["psrType"],
                "quantity": item["quantity"],
            })

    elexon_raw = pd.DataFrame(records)

    if elexon_raw.empty:
        raise ValueError("No Elexon data returned for requested range.")

    elexon_raw["startTime"] = pd.to_datetime(
        elexon_raw["startTime"], utc=True
    ).dt.tz_localize(None)

    elexon_df = (
        elexon_raw.pivot_table(
            index="startTime",
            columns="psrType",
            values="quantity",
            aggfunc="sum",
        )
        .sort_index()
        .reset_index()
    )

    elexon_df.columns.name = None

    fuel_cols = [c for c in elexon_df.columns if c != "startTime"]
    elexon_df["total_output_MW"] = elexon_df[fuel_cols].sum(axis=1)

    # -----------------------------
    # 3) Reindex Elexon to weather timestamps
    # -----------------------------
    elexon_hist_df = (
        elexon_df.set_index("startTime")
        .reindex(weather_index)
        .rename_axis("startTime")
        .reset_index()
    )

    # -----------------------------
    # 4) Fill missing Elexon values
    # -----------------------------
    gen_cols = [c for c in elexon_hist_df.columns if c != "startTime"]

    # interpolate interior gaps
    elexon_hist_df[gen_cols] = elexon_hist_df[gen_cols].interpolate(
        method="linear",
        limit_direction="both"
    )

    # forward fill any remaining trailing gaps
    elexon_hist_df[gen_cols] = elexon_hist_df[gen_cols].ffill()

    # backfill any remaining leading gaps
    elexon_hist_df[gen_cols] = elexon_hist_df[gen_cols].bfill()

    return weather_hist_df, elexon_hist_df

In [6]:
weather_df, elexon_df = get_aligned_weather_elexon_fill()

print(weather_df.shape)
print(elexon_df.shape)

print(weather_df["time"].min(), weather_df["time"].max())
print(elexon_df["startTime"].min(), elexon_df["startTime"].max())

(336, 10)
(336, 13)
2026-03-10 23:00:00 2026-03-17 22:30:00
2026-03-10 23:00:00 2026-03-17 22:30:00


In [7]:
weather_df

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-10 23:00:00,10.50,11.30,100.0,0.0,0.0,0.0,1008.90,0.20,10.5850
1,2026-03-10 23:30:00,10.35,11.70,100.0,0.0,0.0,0.0,1008.45,0.50,10.9075
2,2026-03-11 00:00:00,10.20,12.10,100.0,0.0,0.0,0.0,1008.00,0.80,11.2300
3,2026-03-11 00:30:00,10.05,12.45,100.0,0.0,0.0,0.0,1007.55,0.55,11.3050
4,2026-03-11 01:00:00,9.90,12.80,100.0,0.0,0.0,0.0,1007.10,0.30,11.3800
...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 20:30:00,13.25,4.60,0.0,0.0,0.0,0.0,1014.95,0.00,4.9125
332,2026-03-17 21:00:00,13.10,4.80,0.0,0.0,0.0,0.0,1015.10,0.00,4.9950
333,2026-03-17 21:30:00,12.85,4.20,0.0,0.0,0.0,0.0,1015.15,0.00,5.1800
334,2026-03-17 22:00:00,12.60,3.60,0.0,0.0,0.0,0.0,1015.20,0.00,5.3650


In [8]:
weather_df.isnull().sum()

time                   0
temperature_2m         0
wind_gusts_10m         0
cloud_cover            0
direct_radiation       0
diffuse_radiation      0
shortwave_radiation    0
pressure_msl           0
precipitation          0
wind_speed_100m        0
dtype: int64

In [9]:
elexon_df

,startTime,Biomass,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
0,2026-03-10 23:00:00,1731.0,4533.0,0.0,0.0,0.0,251.0,2851.0,189.0,0.0,11868.229,10187.685,31610.914
1,2026-03-10 23:30:00,1730.0,3932.0,0.0,0.0,0.0,245.0,2854.0,492.0,0.0,11978.611,10065.052,31296.663
2,2026-03-11 00:00:00,1731.0,3334.0,0.0,0.0,1.0,243.0,2848.0,296.0,0.0,12259.452,9897.642,30610.094
3,2026-03-11 00:30:00,1731.0,3314.0,0.0,0.0,0.0,262.0,2851.0,331.0,0.0,12202.053,10001.939,30692.992
4,2026-03-11 01:00:00,2014.0,3605.0,0.0,0.0,0.0,260.0,2852.0,194.0,0.0,12104.196,9915.019,30944.215
...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 20:30:00,2760.0,8885.0,0.0,0.0,0.0,330.0,3632.0,472.0,0.0,8249.009,7850.069,32178.078
332,2026-03-17 21:00:00,2742.0,7386.0,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
333,2026-03-17 21:30:00,2742.0,7386.0,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
334,2026-03-17 22:00:00,2742.0,7386.0,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878


In [10]:
elexon_df.isnull().sum()

startTime                          0
Biomass                            0
Fossil Gas                         0
Fossil Hard coal                   0
Fossil Oil                         0
Hydro Pumped Storage               0
Hydro Run-of-river and poundage    0
Nuclear                            0
Other                              0
Solar                              0
Wind Offshore                      0
Wind Onshore                       0
total_output_MW                    0
dtype: int64

In [11]:
def merge_weather_elexon(weather_df, elexon_df):
    df = weather_df.merge(
        elexon_df,
        left_on="time",
        right_on="startTime",
        how="inner"
    )

    # Drop duplicate time column from Elexon
    df = df.drop(columns=["startTime"])

    return df

In [12]:
weather_df, elexon_df = get_aligned_weather_elexon_fill()

merged_df = merge_weather_elexon(weather_df, elexon_df)

print(merged_df.shape)

(336, 22)


In [13]:
merged_df.tail()

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m,...,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
331,2026-03-17 20:30:00,13.25,4.6,0.0,0.0,0.0,0.0,1014.95,0.0,4.9125,...,0.0,0.0,0.0,330.0,3632.0,472.0,0.0,8249.009,7850.069,32178.078
332,2026-03-17 21:00:00,13.10,4.8,0.0,0.0,0.0,0.0,1015.10,0.0,4.9950,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
333,2026-03-17 21:30:00,12.85,4.2,0.0,0.0,0.0,0.0,1015.15,0.0,5.1800,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
334,2026-03-17 22:00:00,12.60,3.6,0.0,0.0,0.0,0.0,1015.20,0.0,5.3650,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
335,2026-03-17 22:30:00,12.55,4.1,0.0,0.0,0.0,0.0,1015.30,0.0,5.4050,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878


In [14]:
merged_df.isnull().sum()

time                               0
temperature_2m                     0
wind_gusts_10m                     0
cloud_cover                        0
direct_radiation                   0
diffuse_radiation                  0
shortwave_radiation                0
pressure_msl                       0
precipitation                      0
wind_speed_100m                    0
Biomass                            0
Fossil Gas                         0
Fossil Hard coal                   0
Fossil Oil                         0
Hydro Pumped Storage               0
Hydro Run-of-river and poundage    0
Nuclear                            0
Other                              0
Solar                              0
Wind Offshore                      0
Wind Onshore                       0
total_output_MW                    0
dtype: int64

In [15]:
merged_df.columns

Index(['time', 'temperature_2m', 'wind_gusts_10m', 'cloud_cover',
       'direct_radiation', 'diffuse_radiation', 'shortwave_radiation',
       'pressure_msl', 'precipitation', 'wind_speed_100m', 'Biomass',
       'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil', 'Hydro Pumped Storage',
       'Hydro Run-of-river and poundage', 'Nuclear', 'Other', 'Solar',
       'Wind Offshore', 'Wind Onshore', 'total_output_MW'],
      dtype='object')

## scaling

In [16]:
from google.cloud import bigquery

PROJECT = "gridzero-489711"
DATASET = "merged_set"
TABLE = "full_feature_engineered_data_test"

query = f"""
    SELECT *
    FROM {PROJECT}.{DATASET}.{TABLE}
"""

client = bigquery.Client('gridzero-489711')
query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()

/Users/jamesla/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [17]:
feature_cols = [
    # weather
    'temperature_2m_c',
    'wind_speed_100m_ms',
    'wind_gusts_10m_ms',
    'cloud_cover_pct',
    'shortwave_radiation_wm2',
    'direct_radiation_wm2',
    'diffuse_radiation_wm2',
    'pressure_msl_hpa',
    'precipitation_mm',

    # time
    'hour_sin','hour_cos',
    'dow_sin','dow_cos',
    'doy_sin','doy_cos',
    
    # past generation (important)
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [18]:
target_cols = [
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [19]:
from sklearn.preprocessing import MinMaxScaler

X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X = X_scaler.fit_transform(df[feature_cols])
y = y_scaler.fit_transform(df[target_cols])

# prediction

In [20]:
merged_df.dtypes

time                               datetime64[ns]
temperature_2m                            float64
wind_gusts_10m                            float64
cloud_cover                               float64
direct_radiation                          float64
diffuse_radiation                         float64
shortwave_radiation                       float64
pressure_msl                              float64
precipitation                             float64
wind_speed_100m                           float64
Biomass                                   float64
Fossil Gas                                float64
Fossil Hard coal                          float64
Fossil Oil                                float64
Hydro Pumped Storage                      float64
Hydro Run-of-river and poundage           float64
Nuclear                                   float64
Other                                     float64
Solar                                     float64
Wind Offshore                             float64


In [21]:
from tensorflow import keras

model = keras.models.load_model("gs://grid_zero_bucket/lstm_model1.keras")

In [22]:
import numpy as np

def preproc(df):
    if "time" in df.columns:
        df["time"] = df["time"].astype("datetime64[us]",  errors="ignore")
    
        df = df.rename(columns={'time': 'datetime'},  errors="ignore")
    
    weather_cols = [
        'temperature_2m_c','wind_speed_100m_ms','wind_gusts_10m_ms',
        'cloud_cover_pct','shortwave_radiation_wm2','direct_radiation_wm2',
        'diffuse_radiation_wm2','pressure_msl_hpa'
        ]

    gen_cols = [
    'Biomass','Fossil Gas','Fossil Hard coal','Fossil Oil',
    'Hydro Pumped Storage','Hydro Run-of-river and poundage',
    'Nuclear','Other','Solar','Wind Offshore','Wind Onshore'
    ]

    df = df.drop(columns=['Fossil Oil', 'total_output_MW'], errors="ignore")

    # Create Time Features
    if "datetime" in df.columns:
        df['hour'] = df['datetime'].dt.hour
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['day_of_year'] = df['datetime'].dt.dayofyear

    # cyclical encoding
        df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
        df['doy_sin'] = np.sin(2*np.pi*df['day_of_year']/365)
        df['doy_cos'] = np.cos(2*np.pi*df['day_of_year']/365)

    df = df.drop(columns=['hour','day_of_week', 'day_of_year', 'datetime'], errors="ignore")

    # using same naming convention for all columns
    df.columns = (
        df.columns
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('-', '_')
    )

    df = df.rename(columns={
        "temperature_2m": "temperature_2m_c",
        "wind_speed_100m": "wind_speed_100m_ms",
        "wind_gusts_10m": "wind_gusts_10m_ms",
        "cloud_cover": "cloud_cover_pct",
        "shortwave_radiation": "shortwave_radiation_wm2",
        "direct_radiation": "direct_radiation_wm2",
        "diffuse_radiation": "diffuse_radiation_wm2",
        "pressure_msl": "pressure_msl_hpa",
        "precipitation": "precipitation_mm",
    })

    return df

In [23]:
feature_cols = [
    # weather
    'temperature_2m_c',
    'wind_speed_100m_ms',
    'wind_gusts_10m_ms',
    'cloud_cover_pct',
    'shortwave_radiation_wm2',
    'direct_radiation_wm2',
    'diffuse_radiation_wm2',
    'pressure_msl_hpa',
    'precipitation_mm',

    # time
    'hour_sin','hour_cos',
    'dow_sin','dow_cos',
    'doy_sin','doy_cos',
    
    # past generation (important)
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [24]:
df_processed = preproc(merged_df)

In [25]:
df_processed.tail()

,temperature_2m_c,wind_gusts_10m_ms,cloud_cover_pct,direct_radiation_wm2,diffuse_radiation_wm2,shortwave_radiation_wm2,pressure_msl_hpa,precipitation_mm,wind_speed_100m_ms,biomass,...,other,solar,wind_offshore,wind_onshore,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos
331,13.25,4.6,0.0,0.0,0.0,0.0,1014.95,0.0,4.9125,2760.0,...,472.0,0.0,8249.009,7850.069,-0.866025,0.500000,0.781831,0.62349,0.96574,0.259512
332,13.10,4.8,0.0,0.0,0.0,0.0,1015.10,0.0,4.9950,2742.0,...,429.0,0.0,8420.030,7607.848,-0.707107,0.707107,0.781831,0.62349,0.96574,0.259512
333,12.85,4.2,0.0,0.0,0.0,0.0,1015.15,0.0,5.1800,2742.0,...,429.0,0.0,8420.030,7607.848,-0.707107,0.707107,0.781831,0.62349,0.96574,0.259512
334,12.60,3.6,0.0,0.0,0.0,0.0,1015.20,0.0,5.3650,2742.0,...,429.0,0.0,8420.030,7607.848,-0.500000,0.866025,0.781831,0.62349,0.96574,0.259512
335,12.55,4.1,0.0,0.0,0.0,0.0,1015.30,0.0,5.4050,2742.0,...,429.0,0.0,8420.030,7607.848,-0.500000,0.866025,0.781831,0.62349,0.96574,0.259512


In [26]:
df_processed.isnull().sum()

temperature_2m_c                   0
wind_gusts_10m_ms                  0
cloud_cover_pct                    0
direct_radiation_wm2               0
diffuse_radiation_wm2              0
shortwave_radiation_wm2            0
pressure_msl_hpa                   0
precipitation_mm                   0
wind_speed_100m_ms                 0
biomass                            0
fossil_gas                         0
fossil_hard_coal                   0
hydro_pumped_storage               0
hydro_run_of_river_and_poundage    0
nuclear                            0
other                              0
solar                              0
wind_offshore                      0
wind_onshore                       0
hour_sin                           0
hour_cos                           0
dow_sin                            0
dow_cos                            0
doy_sin                            0
doy_cos                            0
dtype: int64

In [27]:
df_processed.shape

(336, 25)

In [28]:
df_processed.columns

Index(['temperature_2m_c', 'wind_gusts_10m_ms', 'cloud_cover_pct',
       'direct_radiation_wm2', 'diffuse_radiation_wm2',
       'shortwave_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm',
       'wind_speed_100m_ms', 'biomass', 'fossil_gas', 'fossil_hard_coal',
       'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear',
       'other', 'solar', 'wind_offshore', 'wind_onshore', 'hour_sin',
       'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos'],
      dtype='object')

In [29]:
feature_cols

['temperature_2m_c',
 'wind_speed_100m_ms',
 'wind_gusts_10m_ms',
 'cloud_cover_pct',
 'shortwave_radiation_wm2',
 'direct_radiation_wm2',
 'diffuse_radiation_wm2',
 'pressure_msl_hpa',
 'precipitation_mm',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'doy_sin',
 'doy_cos',
 'biomass',
 'fossil_gas',
 'fossil_hard_coal',
 'hydro_pumped_storage',
 'hydro_run_of_river_and_poundage',
 'nuclear',
 'other',
 'solar',
 'wind_offshore',
 'wind_onshore']

In [30]:
df_processed = df_processed[feature_cols]

In [31]:
# df_processed_scaled = X_scaler.transform(df_processed)

In [32]:
def make_lstm_input(df):
    """
    Convert a preprocessed dataframe into one LSTM input sample.

    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed dataframe.
    feature_columns : list[str]
        Exact feature order used during model training.
    scaler : fitted scaler, optional
        The same scaler used during training.
    lookback : int
        Number of timesteps expected by the model.

    Returns
    -------
    X : np.ndarray
        Shape (1, lookback, n_features)
    """

    # Ensure required columns exist
    missing = [col for col in feature_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep exact training column order
    df = df[feature_cols]

    # Handle missing values if needed
    df = df.ffill().bfill()

    if len(df) < 336:
        raise ValueError(f"Need at least {336} rows, got {len(df)}")

    # Take latest 336 rows
    df_window = df.iloc[-336:]

    # Apply training scaler if provided
    values = df_window.values

    values = X_scaler.transform(df_window)

    # Convert to LSTM shape: (1, 336, n_features)
    X = np.expand_dims(values.astype(np.float32), axis=0)

    return X

In [33]:
df_processed.columns

Index(['temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms',
       'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2',
       'diffuse_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos',
       'biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage',
       'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
       'wind_offshore', 'wind_onshore'],
      dtype='object')

In [34]:
X_input = make_lstm_input(df=df_processed)

print(X_input.shape)
# (1, 336, n_features)

(1, 336, 25)


In [35]:
y_pred = model.predict(X_input)
y_pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step


array([[0.77087677, 0.24219954, 0.00376548, 0.04028987, 0.21577163,
        0.44404405, 0.09598617, 0.0113602 , 0.6007933 , 0.63650453]],
      dtype=float32)

In [36]:
y_pred = y_scaler.inverse_transform(y_pred)
y_pred

array([[2644.8782 , 6617.618  ,   42.71184,   99.63684,  307.2588 ,
        3714.8726 ,  357.3565 ,  153.96483, 7907.8477 , 7157.2637 ]],
      dtype=float32)

# Predicting multiple days now

In [37]:
import requests
import pandas as pd


def get_london_forecast_step_halfhour(step=0):
    """
    Return a single-row DataFrame for London's forecast at a given half-hour step.

    Parameters
    ----------
    step : int, default=0
        Half-hour step from the most recent forecast boundary.
        0 = most recent forecast rounded down to nearest half hour
        1 = next half-hour forecast
        2 = half-hour after that
        ...
        Up to 14 days ahead.

    Returns
    -------
    pd.DataFrame
        Single-row DataFrame containing one forecast record.
    """
    if step < 0:
        raise ValueError("step must be >= 0")

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation",
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "wind_speed_unit": "ms",
        "current": "temperature_2m",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])

    # Create wind_speed_100m and drop originals
    df["wind_speed_100m"] = (df["wind_speed_120m"] + df["wind_speed_80m"]) / 2
    df = df.drop(columns=["wind_speed_120m", "wind_speed_80m"])

    # Convert hourly data to half-hourly
    df = (
        df.set_index("time")
        .resample("30min")
        .interpolate(method="time")
        .reset_index()
    )

    # Most recent forecast boundary rounded down to nearest half hour
    forecast_start = pd.to_datetime(data["current"]["time"]).floor("30min")

    # Keep only forecast rows from that point onward
    forecast_df = df[df["time"] >= forecast_start].reset_index(drop=True)

    max_step = len(forecast_df) - 1
    if step > max_step:
        raise IndexError(f"step must be between 0 and {max_step}")

    # Return a single-record DataFrame
    return forecast_df.iloc[[step]].reset_index(drop=True)

In [38]:
latest_forecast = get_london_forecast_step_halfhour()
latest_forecast

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-17 22:30:00,12.55,4.1,0.0,0.0,0.0,0.0,1015.3,0.0,5.405


In [39]:
test_weather_df, test_elexon_df = get_aligned_weather_elexon_fill()
test_weather_df

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-10 23:00:00,10.50,11.30,100.0,0.0,0.0,0.0,1008.90,0.20,10.5850
1,2026-03-10 23:30:00,10.35,11.70,100.0,0.0,0.0,0.0,1008.45,0.50,10.9075
2,2026-03-11 00:00:00,10.20,12.10,100.0,0.0,0.0,0.0,1008.00,0.80,11.2300
3,2026-03-11 00:30:00,10.05,12.45,100.0,0.0,0.0,0.0,1007.55,0.55,11.3050
4,2026-03-11 01:00:00,9.90,12.80,100.0,0.0,0.0,0.0,1007.10,0.30,11.3800
...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 20:30:00,13.25,4.60,0.0,0.0,0.0,0.0,1014.95,0.00,4.9125
332,2026-03-17 21:00:00,13.10,4.80,0.0,0.0,0.0,0.0,1015.10,0.00,4.9950
333,2026-03-17 21:30:00,12.85,4.20,0.0,0.0,0.0,0.0,1015.15,0.00,5.1800
334,2026-03-17 22:00:00,12.60,3.60,0.0,0.0,0.0,0.0,1015.20,0.00,5.3650


In [40]:
y_pred = model.predict(X_input)
y_pred = y_scaler.inverse_transform(y_pred)
y_pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


array([[2644.8782 , 6617.618  ,   42.71184,   99.63684,  307.2588 ,
        3714.8726 ,  357.3565 ,  153.96483, 7907.8477 , 7157.2637 ]],
      dtype=float32)

In [41]:
target_cols

['biomass',
 'fossil_gas',
 'fossil_hard_coal',
 'hydro_pumped_storage',
 'hydro_run_of_river_and_poundage',
 'nuclear',
 'other',
 'solar',
 'wind_offshore',
 'wind_onshore']

In [42]:
pred_df = pd.DataFrame(y_pred, columns=target_cols)
pred_df

,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,2644.878174,6617.618164,42.711842,99.636841,307.258789,3714.872559,357.356506,153.964828,7907.847656,7157.263672


In [43]:
final_df = pd.concat([latest_forecast.reset_index(drop=True),
                      pred_df.reset_index(drop=True)], axis=1)

In [44]:
final_df

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,2026-03-17 22:30:00,12.55,4.1,0.0,0.0,0.0,0.0,1015.3,0.0,5.405,2644.878174,6617.618164,42.711842,99.636841,307.258789,3714.872559,357.356506,153.964828,7907.847656,7157.263672


In [45]:
new_data_df = preproc(final_df)
new_data_df

,temperature_2m_c,wind_gusts_10m_ms,cloud_cover_pct,direct_radiation_wm2,diffuse_radiation_wm2,shortwave_radiation_wm2,pressure_msl_hpa,precipitation_mm,wind_speed_100m_ms,biomass,...,other,solar,wind_offshore,wind_onshore,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos
0,12.55,4.1,0.0,0.0,0.0,0.0,1015.3,0.0,5.405,2644.878174,...,357.356506,153.964828,7907.847656,7157.263672,-0.5,0.866025,0.781831,0.62349,0.96574,0.259512


# Looping to predict multiple days

In [46]:
merged_df.head()

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m,...,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
0,2026-03-10 23:00:00,10.50,11.30,100.0,0.0,0.0,0.0,1008.90,0.20,10.5850,...,0.0,0.0,0.0,251.0,2851.0,189.0,0.0,11868.229,10187.685,31610.914
1,2026-03-10 23:30:00,10.35,11.70,100.0,0.0,0.0,0.0,1008.45,0.50,10.9075,...,0.0,0.0,0.0,245.0,2854.0,492.0,0.0,11978.611,10065.052,31296.663
2,2026-03-11 00:00:00,10.20,12.10,100.0,0.0,0.0,0.0,1008.00,0.80,11.2300,...,0.0,0.0,1.0,243.0,2848.0,296.0,0.0,12259.452,9897.642,30610.094
3,2026-03-11 00:30:00,10.05,12.45,100.0,0.0,0.0,0.0,1007.55,0.55,11.3050,...,0.0,0.0,0.0,262.0,2851.0,331.0,0.0,12202.053,10001.939,30692.992
4,2026-03-11 01:00:00,9.90,12.80,100.0,0.0,0.0,0.0,1007.10,0.30,11.3800,...,0.0,0.0,0.0,260.0,2852.0,194.0,0.0,12104.196,9915.019,30944.215


In [47]:
model = keras.models.load_model("gs://grid_zero_bucket/lstm_model1.keras")

In [103]:
df_processed = preproc(merged_df)

for i in range(671+1):
    df_processed = df_processed[feature_cols]

    X_input = make_lstm_input(df=df_processed)

    y_pred = model.predict(X_input)
    y_pred = y_scaler.inverse_transform(y_pred)
    
    pred_df = pd.DataFrame(y_pred, columns=target_cols)
    latest_forecast = get_london_forecast_step_halfhour(i)

    new_data = pd.concat([latest_forecast.reset_index(drop=True),
                  pred_df.reset_index(drop=True)], axis=1)

    new_row = preproc(new_data)
    new_row = new_row[feature_cols]

    df_processed = pd.concat([df_processed, new_row], ignore_index=True)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━

KeyboardInterrupt: 

# testing

In [49]:
merged_df

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m,...,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
0,2026-03-10 23:00:00,10.50,11.30,100.0,0.0,0.0,0.0,1008.90,0.20,10.5850,...,0.0,0.0,0.0,251.0,2851.0,189.0,0.0,11868.229,10187.685,31610.914
1,2026-03-10 23:30:00,10.35,11.70,100.0,0.0,0.0,0.0,1008.45,0.50,10.9075,...,0.0,0.0,0.0,245.0,2854.0,492.0,0.0,11978.611,10065.052,31296.663
2,2026-03-11 00:00:00,10.20,12.10,100.0,0.0,0.0,0.0,1008.00,0.80,11.2300,...,0.0,0.0,1.0,243.0,2848.0,296.0,0.0,12259.452,9897.642,30610.094
3,2026-03-11 00:30:00,10.05,12.45,100.0,0.0,0.0,0.0,1007.55,0.55,11.3050,...,0.0,0.0,0.0,262.0,2851.0,331.0,0.0,12202.053,10001.939,30692.992
4,2026-03-11 01:00:00,9.90,12.80,100.0,0.0,0.0,0.0,1007.10,0.30,11.3800,...,0.0,0.0,0.0,260.0,2852.0,194.0,0.0,12104.196,9915.019,30944.215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 20:30:00,13.25,4.60,0.0,0.0,0.0,0.0,1014.95,0.00,4.9125,...,0.0,0.0,0.0,330.0,3632.0,472.0,0.0,8249.009,7850.069,32178.078
332,2026-03-17 21:00:00,13.10,4.80,0.0,0.0,0.0,0.0,1015.10,0.00,4.9950,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
333,2026-03-17 21:30:00,12.85,4.20,0.0,0.0,0.0,0.0,1015.15,0.00,5.1800,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878
334,2026-03-17 22:00:00,12.60,3.60,0.0,0.0,0.0,0.0,1015.20,0.00,5.3650,...,0.0,0.0,113.0,331.0,3646.0,429.0,0.0,8420.030,7607.848,30674.878


In [50]:
merged_df.shape

(336, 22)

In [51]:
merged_df.isnull().sum()

time                               0
temperature_2m                     0
wind_gusts_10m                     0
cloud_cover                        0
direct_radiation                   0
diffuse_radiation                  0
shortwave_radiation                0
pressure_msl                       0
precipitation                      0
wind_speed_100m                    0
Biomass                            0
Fossil Gas                         0
Fossil Hard coal                   0
Fossil Oil                         0
Hydro Pumped Storage               0
Hydro Run-of-river and poundage    0
Nuclear                            0
Other                              0
Solar                              0
Wind Offshore                      0
Wind Onshore                       0
total_output_MW                    0
dtype: int64

In [52]:
df_processed = preproc(merged_df)
df_processed.head(1)

,temperature_2m_c,wind_gusts_10m_ms,cloud_cover_pct,direct_radiation_wm2,diffuse_radiation_wm2,shortwave_radiation_wm2,pressure_msl_hpa,precipitation_mm,wind_speed_100m_ms,biomass,...,other,solar,wind_offshore,wind_onshore,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos
0,10.5,11.3,100.0,0.0,0.0,0.0,1008.9,0.2,10.585,1731.0,...,189.0,0.0,11868.229,10187.685,-0.258819,0.965926,0.781831,0.62349,0.927542,0.37372


In [53]:
df_processed = df_processed[feature_cols]
df_processed.head(1)

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,hour_sin,...,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,10.5,10.585,11.3,100.0,0.0,0.0,0.0,1008.9,0.2,-0.258819,...,1731.0,4533.0,0.0,0.0,251.0,2851.0,189.0,0.0,11868.229,10187.685


In [64]:
df_processed.columns

Index(['temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms',
       'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2',
       'diffuse_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos',
       'biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage',
       'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
       'wind_offshore', 'wind_onshore'],
      dtype='object')

In [63]:
df_processed.shape

(336, 25)

In [54]:
X_input = make_lstm_input(df=df_processed)
X_input.shape

(1, 336, 25)

In [55]:
y_pred = model.predict(X_input)
y_pred = y_scaler.inverse_transform(y_pred)
y_pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step


array([[2644.8782 , 6617.618  ,   42.71184,   99.63684,  307.2588 ,
        3714.8726 ,  357.3565 ,  153.96483, 7907.8477 , 7157.2637 ]],
      dtype=float32)

In [56]:
pred_df = pd.DataFrame(y_pred, columns=target_cols)
pred_df

,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,2644.878174,6617.618164,42.711842,99.636841,307.258789,3714.872559,357.356506,153.964828,7907.847656,7157.263672


In [65]:
pred_df.shape

(1, 10)

In [67]:
pred_df.columns

Index(['biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage',
       'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
       'wind_offshore', 'wind_onshore'],
      dtype='object')

In [90]:
latest_forecast = get_london_forecast_step_halfhour(0)
latest_forecast

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-17 23:00:00,12.5,4.6,0.0,0.0,0.0,0.0,1015.4,0.0,5.445


In [91]:
latest_forecast.shape

(1, 10)

In [92]:
latest_forecast.columns

Index(['time', 'temperature_2m', 'wind_gusts_10m', 'cloud_cover',
       'direct_radiation', 'diffuse_radiation', 'shortwave_radiation',
       'pressure_msl', 'precipitation', 'wind_speed_100m'],
      dtype='object')

In [93]:
new_data = pd.concat([latest_forecast.reset_index(drop=True),
                  pred_df.reset_index(drop=True)], axis=1)
new_data

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,2026-03-17 23:00:00,12.5,4.6,0.0,0.0,0.0,0.0,1015.4,0.0,5.445,2644.878174,6617.618164,42.711842,99.636841,307.258789,3714.872559,357.356506,153.964828,7907.847656,7157.263672


In [94]:
# merged_df.columns

In [95]:
# merged_df.shape

In [96]:
# merged_df.columns

In [97]:
new_row.columns

Index(['time', 'temperature_2m', 'wind_gusts_10m', 'cloud_cover',
       'direct_radiation', 'diffuse_radiation', 'shortwave_radiation',
       'pressure_msl', 'precipitation', 'wind_speed_100m', 'biomass',
       'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage',
       'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
       'wind_offshore', 'wind_onshore'],
      dtype='object')

In [98]:
new_row = preproc(new_data)
new_row.shape

(1, 25)

In [99]:
new_row

,temperature_2m_c,wind_gusts_10m_ms,cloud_cover_pct,direct_radiation_wm2,diffuse_radiation_wm2,shortwave_radiation_wm2,pressure_msl_hpa,precipitation_mm,wind_speed_100m_ms,biomass,...,other,solar,wind_offshore,wind_onshore,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos
0,12.5,4.6,0.0,0.0,0.0,0.0,1015.4,0.0,5.445,2644.878174,...,357.356506,153.964828,7907.847656,7157.263672,-0.258819,0.965926,0.781831,0.62349,0.96574,0.259512


In [100]:
new_row = new_row[feature_cols]
new_row

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,hour_sin,...,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,12.5,5.445,4.6,0.0,0.0,0.0,0.0,1015.4,0.0,-0.258819,...,2644.878174,6617.618164,42.711842,99.636841,307.258789,3714.872559,357.356506,153.964828,7907.847656,7157.263672


In [101]:
df_processed.columns

Index(['temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms',
       'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2',
       'diffuse_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos',
       'biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage',
       'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
       'wind_offshore', 'wind_onshore'],
      dtype='object')

In [102]:
df = pd.concat([df_processed, new_row], ignore_index=True)
df

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,hour_sin,...,biomass,fossil_gas,fossil_hard_coal,hydro_pumped_storage,hydro_run_of_river_and_poundage,nuclear,other,solar,wind_offshore,wind_onshore
0,10.50,10.5850,11.30,100.0,0.0,0.0,0.0,1008.90,0.20,-0.258819,...,1731.000000,4533.000000,0.000000,0.000000,251.000000,2851.000000,189.000000,0.000000,11868.229000,10187.685000
1,10.35,10.9075,11.70,100.0,0.0,0.0,0.0,1008.45,0.50,-0.258819,...,1730.000000,3932.000000,0.000000,0.000000,245.000000,2854.000000,492.000000,0.000000,11978.611000,10065.052000
2,10.20,11.2300,12.10,100.0,0.0,0.0,0.0,1008.00,0.80,0.000000,...,1731.000000,3334.000000,0.000000,1.000000,243.000000,2848.000000,296.000000,0.000000,12259.452000,9897.642000
3,10.05,11.3050,12.45,100.0,0.0,0.0,0.0,1007.55,0.55,0.000000,...,1731.000000,3314.000000,0.000000,0.000000,262.000000,2851.000000,331.000000,0.000000,12202.053000,10001.939000
4,9.90,11.3800,12.80,100.0,0.0,0.0,0.0,1007.10,0.30,0.258819,...,2014.000000,3605.000000,0.000000,0.000000,260.000000,2852.000000,194.000000,0.000000,12104.196000,9915.019000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332,13.10,4.9950,4.80,0.0,0.0,0.0,0.0,1015.10,0.00,-0.707107,...,2742.000000,7386.000000,0.000000,113.000000,331.000000,3646.000000,429.000000,0.000000,8420.030000,7607.848000
333,12.85,5.1800,4.20,0.0,0.0,0.0,0.0,1015.15,0.00,-0.707107,...,2742.000000,7386.000000,0.000000,113.000000,331.000000,3646.000000,429.000000,0.000000,8420.030000,7607.848000
334,12.60,5.3650,3.60,0.0,0.0,0.0,0.0,1015.20,0.00,-0.500000,...,2742.000000,7386.000000,0.000000,113.000000,331.000000,3646.000000,429.000000,0.000000,8420.030000,7607.848000
335,12.55,5.4050,4.10,0.0,0.0,0.0,0.0,1015.30,0.00,-0.500000,...,2742.000000,7386.000000,0.000000,113.000000,331.000000,3646.000000,429.000000,0.000000,8420.030000,7607.848000
